In [95]:
from typing import TypedDict, Annotated

from IPython.core.magics import config
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, BaseMessage, AIMessage
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END, state
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_tavily import TavilySearch

load_dotenv()

True

In [96]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [97]:
llm = ChatGroq(model="qwen/qwen3-32b", temperature=0.4, reasoning_format="hidden")

search_tool = TavilySearch(
    max_results=3,
    topic="general",
    # include_answer=False,
    # include_raw_content=False,
    include_images=False,
    # include_image_descriptions=False,
    search_depth="advanced",
    time_range="day",
    # include_domains=None,
    # exclude_domains=None,
    # country=None
    # include_favicon=False
    # include_usage=False
)
tool_list = [search_tool]
llm_with_tool = llm.bind_tools(tool_list)

In [98]:
def chat_node(state: ChatState) -> ChatState:
    # print(list(state["messages"]))
    response_llm = llm_with_tool.invoke(input=state["messages"])
    return {"messages": [response_llm]}

In [99]:
tools_node = ToolNode(tool_list)

In [100]:
checkpointer = InMemorySaver()
graph = StateGraph(ChatState)

graph.add_node("chat_node", chat_node)
graph.add_node("tools", tools_node)

graph.add_edge(START, "chat_node")
graph.add_conditional_edges("chat_node", tools_condition)
graph.add_edge("tools", "chat_node")

workflow = graph.compile(checkpointer=checkpointer)

In [101]:
thread_id = "mychat-ui-2"
config = {"configurable": {"thread_id": thread_id}}

while True:
    user_message = input("Human: ")
    initial_state = {"messages": [HumanMessage(content=user_message)]}

    print(f"Human: {user_message}")
    if user_message.lower().strip() in ["bye", "exit", "quit"]:
        break

    response = workflow.invoke(input= initial_state, config= config)
    print(f"AI: {response["messages"][-1].content} \n")

Human: Who won the fifa world cup match yesterday ?
AI: It appears there were no official FIFA World Cup matches held on October 27, 2023. The 2022 World Cup has concluded, and the next edition (2026) begins in June 2026. The search results referenced historical qualifiers (1981) and future 2026 World Cup recaps, but no October 27, 2023 matches were found. You might want to verify the date or check recent FIFA qualifiers/friendly matches for updates. 



KeyboardInterrupt: Interrupted by user

In [ ]:
# print("=== State History ===")
# for snapshot in workflow.get_state_history(config):
#     print(f"--- Step {snapshot.metadata['step']} | {snapshot.created_at} ---")
#     print("Next:  ", snapshot.next)
#     print("Values:", {k: v[:60] + "..." if isinstance(v, str) and len(v) > 60 else v
#                      for k, v in snapshot.values.items()})
#     print()